<a href="https://colab.research.google.com/github/DangHuuLong/Ai-Recruiter-Mini-Ai-Service/blob/experiment%2Fcross-encoder-v0.3/notebooks/fine_tune_cross_encoder_v0_3_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Cross-Encoder CV-JD v0.3

Fine-tune `cross-encoder/ms-marco-MiniLM-L-12-v2` với **boundary-aware loss** trên 4900 CV-JD pairs.
v0.3 giữ nguyên base model (L-12) từ v0.2, chỉ thay loss function để trực tiếp tối ưu label accuracy.

| | |
|---|---|
| **Dataset** | v0.3 — 4900 train / 1050 validation / 1050 test |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `BoundaryAwareLoss`: MSE + ordinal BCE tại boundaries 40/60/75/90 |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.3` |

**Mục tiêu**: vượt LabelAcc 60.76% của v0.2 bằng cách penalise predictions sai bucket boundary.

In [ ]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.3
!git pull origin experiment/cross-encoder-v0.3

/content/Ai-Recruiter-Mini-Ai-Service
Already on 'experiment/cross-encoder-v0.3'
Your branch is up to date with 'origin/experiment/cross-encoder-v0.3'.
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 538 bytes | 179.00 KiB/s, done.
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/cross-encoder-v0.3 -> FETCH_HEAD
   3be9368..1306b78  experiment/cross-encoder-v0.3 -> origin/experiment/cross-encoder-v0.3
Updating 3be9368..1306b78
Fast-forward
 requirements.txt | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [ ]:
!pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.3/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")

train       :  4900 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
validation  :  1050 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
test        :  1050 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']


## Debug run — sanity check BoundaryAwareLoss (1 epoch, 40 samples)

In [ ]:
!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --loss boundary \
    --evaluator label_acc \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --epochs 1 \
    --batch-size 4 \
    --max-train-samples 40 \
    --max-eval-samples 20

2026-06-18 16:23:20.205560: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Data dir   : datasets/versions/v0.3/cross_encoder
Output dir : artifacts/models/cross-encoder-cv-jd-v0.1
Loss       : boundary  (BoundaryAwareLoss)
Evaluator  : label_acc
Epochs     : 1  |  Batch size: 4  |  Max length: 512

Train: 40 pairs  |  Val: 20 pairs

Steps/epoch: 10  |  Warmup steps: 1

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:234: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  epoch  1/1  step    10  val_label_acc=0.0500
  [done]  val_label_ac

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


## Full training — 10 epochs, save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"

import os, time
os.makedirs(f"{drive_base}/models/cross-encoder-cv-jd-v0.3", exist_ok=True)
time.sleep(3)

!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --loss boundary \
    --evaluator label_acc \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --output-dir {drive_base}/models/cross-encoder-cv-jd-v0.3 \
    --report-path artifacts/reports/fine_tune_cross_encoder_v0.3_report.json \
    --epochs 10 \
    --batch-size 16

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2026-06-18 16:23:49.136377: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Data dir   : datasets/versions/v0.3/cross_encoder
Output dir : /content/drive/MyDrive/ai-recruiter/models/cross-encoder-cv-jd-v0.3
Loss       : boundary  (BoundaryAwareLoss)
Evaluator  : label_acc
Epochs     : 10  |  Batch size: 16  |  Max length: 512

Train: 4900 pairs  |  Val: 1050 pairs

Steps/epoch: 307  |  Warmup steps: 30

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:234: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Pleas

In [ ]:
import json
from pathlib import Path

report = json.loads(
    Path("artifacts/reports/fine_tune_cross_encoder_v0.3_report.json").read_text(encoding="utf-8")
)
print(f"Base model : {report['base_model']}")
print(f"Loss       : {report['loss']}")
print()
print(json.dumps(report["metrics"], indent=2))

Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Loss       : boundary

{
  "validation": {
    "mae": 11.0196,
    "rmse": 14.9074,
    "label_accuracy": 0.5552,
    "pair_count": 1050,
    "mean_predicted_score": 64.5404,
    "mean_target_score": 60.6457
  },
  "test": {
    "mae": 10.404,
    "rmse": 15.1145,
    "label_accuracy": 0.5962,
    "pair_count": 1050,
    "mean_predicted_score": 64.2474,
    "mean_target_score": 62.8514
  }
}


In [ ]:
import shutil
from pathlib import Path

reports_dir = Path(drive_base) / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(
    "artifacts/reports/fine_tune_cross_encoder_v0.3_report.json",
    reports_dir / "fine_tune_cross_encoder_v0.3_report.json",
)
print(f"Saved to {reports_dir}")

Saved to /content/drive/MyDrive/ai-recruiter/reports
